In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# יצירת תוסף Transpiler

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';





<details>
<summary><b>Package versions</b></summary>

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.0
```
</details>

יצירת [תוסף Transpiler](transpiler-plugins) היא דרך מצוינת לשתף את קוד הטרנספילציה שלך עם קהילת Qiskit הרחבה, ולאפשר למשתמשים אחרים ליהנות מהפונקציונליות שפיתחת. תודה על עניינך בתרומה לקהילת Qiskit!

לפני שאתה יוצר תוסף Transpiler, עליך להחליט איזה סוג תוסף מתאים למצב שלך. קיימים שלושה סוגי תוספי Transpiler:
- [**תוסף שלב Transpiler**](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins). בחר באפשרות זו אם אתה מגדיר pass manager שיכול להחליף אחד מ-[6 השלבים](transpiler-stages) של pass manager מובנה עם שלבים מוגדרים מראש.
- [**תוסף סינתזת יוניטרי**](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin). בחר באפשרות זו אם קוד הטרנספילציה שלך מקבל כקלט מטריצה יוניטרית (המיוצגת כמערך Numpy) ומייצר תיאור של Circuit קוונטי המממש את אותה מטריצה יוניטרית.
- [**תוסף סינתזה ברמה גבוהה**](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin). בחר באפשרות זו אם קוד הטרנספילציה שלך מקבל כקלט "אובייקט ברמה גבוהה" כגון אופרטור Clifford או פונקציה לינארית, ומייצר תיאור של Circuit קוונטי המממש את אותו אובייקט ברמה גבוהה. אובייקטים ברמה גבוהה מיוצגים על ידי תת-מחלקות של מחלקת [Operation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Operation).

לאחר שקבעת איזה סוג תוסף ליצור, בצע את השלבים הבאים ליצירת התוסף:

1. צור תת-מחלקה של מחלקת התוסף המופשטת המתאימה:
   - [PassManagerStagePlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePlugin) לתוסף שלב Transpiler,
   - [UnitarySynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin) לתוסף סינתזת יוניטרי, ו-
   - [HighLevelSynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin) לתוסף סינתזה ברמה גבוהה.
2. חשוף את המחלקה כ-[entry point של setuptools](https://setuptools.pypa.io/en/latest/userguide/entry_point.html) במטא-נתונים של החבילה, בדרך כלל על ידי עריכת קובץ `pyproject.toml`, ‏`setup.cfg`, או `setup.py` של חבילת Python שלך.

אין הגבלה על מספר התוספים שחבילה אחת יכולה להגדיר, אך לכל תוסף חייב להיות שם ייחודי. ה-Qiskit SDK עצמו כולל מספר תוספים, שגם שמותיהם שמורים. השמות השמורים הם:

- תוספי שלב Transpiler: ראה [טבלה זו](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins#plugin-stages).
- תוספי סינתזת יוניטרי: `default`, `aqc`, `sk`
- תוספי סינתזה ברמה גבוהה:

| מחלקת Operation | שם Operation | שמות שמורים |
| --- | --- | --- |
| [Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford#clifford) | `clifford` | `default`, `ag`, `bm`, `greedy`, `layers`, `lnn` |
| [LinearFunction](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction#linearfunction) | `linear_function` | `default`, `kms`, `pmh` |
| [PermutationGate](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.PermutationGate#permutationgate) | `permutation` | `default`, `kms`, `basic`, `acg`, `token_swapper` |

בחלקים הבאים נציג דוגמאות לשלבים אלו עבור סוגי התוספים השונים. בדוגמאות אלו אנו מניחים שאנחנו יוצרים חבילת Python בשם `my_qiskit_plugin`. למידע על יצירת חבילות Python, תוכל לעיין ב-[מדריך זה](https://packaging.python.org/en/latest/tutorials/packaging-projects/) באתר Python.

## דוגמה: יצירת תוסף שלב Transpiler
בדוגמה זו נצור תוסף שלב Transpiler עבור שלב `layout` (ראה [שלבי Transpiler](transpiler-stages) לתיאור 6 השלבים של צינור הטרנספילציה המובנה של Qiskit).
התוסף שלנו פשוט מריץ את [VF2Layout](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.VF2Layout) עם מספר ניסיונות התלוי ברמת האופטימיזציה המבוקשת.

ראשית, ניצור תת-מחלקה של [PassManagerStagePlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePlugin). קיימת שיטה אחת שעלינו לממש, הנקראת [`pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePlugin#pass_manager). שיטה זו מקבלת כקלט [PassManagerConfig](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManagerConfig) ומחזירה את ה-pass manager שאנחנו מגדירים. האובייקט PassManagerConfig מאחסן מידע על ה-Backend המטרה, כגון מפת הצימוד ו-Gate הבסיסיים.

In [1]:
# This import is needed for python versions prior to 3.10
from __future__ import annotations

from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import VF2Layout
from qiskit.transpiler.passmanager_config import PassManagerConfig
from qiskit.transpiler.preset_passmanagers import common
from qiskit.transpiler.preset_passmanagers.plugin import (
    PassManagerStagePlugin,
)


class MyLayoutPlugin(PassManagerStagePlugin):
    def pass_manager(
        self,
        pass_manager_config: PassManagerConfig,
        optimization_level: int | None = None,
    ) -> PassManager:
        layout_pm = PassManager(
            [
                VF2Layout(
                    coupling_map=pass_manager_config.coupling_map,
                    properties=pass_manager_config.backend_properties,
                    max_trials=optimization_level * 10 + 1,
                    target=pass_manager_config.target,
                )
            ]
        )
        layout_pm += common.generate_embed_passmanager(
            pass_manager_config.coupling_map
        )
        return layout_pm

כעת נחשוף את התוסף על ידי הוספת entry point במטא-נתונים של חבילת Python שלנו.
כאן אנו מניחים שהמחלקה שהגדרנו חשופה במודול בשם `my_qiskit_plugin`, לדוגמה על ידי ייבואה בקובץ `__init__.py` של מודול `my_qiskit_plugin`.
נערוך את קובץ `pyproject.toml`, ‏`setup.cfg`, או `setup.py` של החבילה שלנו (בהתאם לסוג הקובץ שבחרת לאחסן את מטא-נתוני פרויקט Python שלך):

<Tabs>
  <TabItem value="package-table-toml" label="pyproject.toml" default>
    ```toml
    [project.entry-points."qiskit.transpiler.layout"]
    "my_layout" = "my_qiskit_plugin:MyLayoutPlugin"
    ```
  </TabItem>
  <TabItem value="package-table-cfg" label="setup.cfg">
    ```ini
    [options.entry_points]
    qiskit.transpiler.layout =
        my_layout = my_qiskit_plugin:MyLayoutPlugin
    ```
  </TabItem>
  <TabItem value="package-table-py" label="setup.py">
    ```python
    from setuptools import setup

    setup(
        # ...,
        entry_points={
            'qiskit.transpiler.layout': [
                'my_layout = my_qiskit_plugin:MyLayoutPlugin',
            ]
        }
    )
    ```
  </TabItem>
</Tabs>

ראה את [טבלת שלבי תוסף Transpiler](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins#stage-table) לנקודות הכניסה והציפיות עבור כל שלב Transpiler.

כדי לבדוק שהתוסף שלך מזוהה בהצלחה על ידי Qiskit, התקן את חבילת התוסף שלך ועקוב אחר ההוראות ב-[תוספי Transpiler](transpiler-plugins#list-available-transpiler-stage-plugins) לרשימת התוספים המותקנים, וודא שהתוסף שלך מופיע ברשימה:

In [2]:
from qiskit.transpiler.preset_passmanagers.plugin import list_stage_plugins

list_stage_plugins("layout")

['default', 'dense', 'sabre', 'trivial']

If our example plugin were installed, then the name `my_layout` would appear in this list.

If you want to use a built-in transpiler stage as the starting point for your transpiler stage plugin, you can obtain the pass manager for a built-in transpiler stage using [PassManagerStagePluginManager](/docs/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePluginManager#passmanagerstagepluginmanager). The following code cell shows how to do this to obtain the built-in optimization stage for optimization level 3.

In [3]:
from qiskit.transpiler.preset_passmanagers.plugin import (
    PassManagerStagePluginManager,
)

# Initialize the plugin manager
plugin_manager = PassManagerStagePluginManager()

# Here we create a pass manager config to use as an example.
# Instead, you should use the pass manager config that you already received as input
# to the pass_manager method of your PassManagerStagePlugin.
pass_manager_config = PassManagerConfig()

# Obtain the desired built-in transpiler stage
optimization = plugin_manager.get_passmanager_stage(
    "optimization", "default", pass_manager_config, optimization_level=3
)

אם תוסף הדוגמה שלנו היה מותקן, אז השם `my_layout` היה מופיע ברשימה זו.

אם ברצונך להשתמש בשלב Transpiler מובנה כנקודת מוצא לתוסף שלב Transpiler שלך, תוכל לקבל את ה-pass manager עבור שלב Transpiler מובנה באמצעות [PassManagerStagePluginManager](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePluginManager#passmanagerstagepluginmanager). תא הקוד הבא מראה כיצד לעשות זאת כדי לקבל את שלב האופטימיזציה המובנה עבור רמת אופטימיזציה 3.

In [4]:
import numpy as np
from qiskit.circuit import QuantumCircuit, QuantumRegister
from qiskit.converters import circuit_to_dag
from qiskit.dagcircuit.dagcircuit import DAGCircuit
from qiskit.quantum_info import Operator
from qiskit.transpiler.passes import UnitarySynthesis
from qiskit.transpiler.passes.synthesis.plugin import UnitarySynthesisPlugin


class MyUnitarySynthesisPlugin(UnitarySynthesisPlugin):
    @property
    def supports_basis_gates(self):
        # Returns True if the plugin can target a list of basis gates
        return True

    @property
    def supports_coupling_map(self):
        # Returns True if the plugin can synthesize for a given coupling map
        return False

    @property
    def supports_natural_direction(self):
        # Returns True if the plugin supports a toggle for considering
        # directionality of 2-qubit gates
        return False

    @property
    def supports_pulse_optimize(self):
        # Returns True if the plugin can optimize pulses during synthesis
        return False

    @property
    def supports_gate_lengths(self):
        # Returns True if the plugin can accept information about gate lengths
        return False

    @property
    def supports_gate_errors(self):
        # Returns True if the plugin can accept information about gate errors
        return False

    @property
    def supports_gate_lengths_by_qubit(self):
        # Returns True if the plugin can accept information about gate lengths
        # (The format of the input differs from supports_gate_lengths)
        return False

    @property
    def supports_gate_errors_by_qubit(self):
        # Returns True if the plugin can accept information about gate errors
        # (The format of the input differs from supports_gate_errors)
        return False

    @property
    def min_qubits(self):
        # Returns the minimum number of qubits the plugin supports
        return None

    @property
    def max_qubits(self):
        # Returns the maximum number of qubits the plugin supports
        return None

    @property
    def supported_bases(self):
        # Returns a dictionary of supported bases for synthesis
        return None

    def run(self, unitary: np.ndarray, **options) -> DAGCircuit:
        basis_gates = options["basis_gates"]
        synth_pass = UnitarySynthesis(basis_gates, min_qubits=3)
        qubits = QuantumRegister(3)
        circuit = QuantumCircuit(qubits)
        circuit.append(Operator(unitary).to_instruction(), qubits)
        dag_circuit = synth_pass.run(circuit_to_dag(circuit))
        return dag_circuit

## דוגמה: יצירת תוסף סינתזת יוניטרי
בדוגמה זו נצור תוסף סינתזת יוניטרי שפשוט משתמש ב-[UnitarySynthesis](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.UnitarySynthesis#unitarysynthesis) המובנה להמרת Gate. כמובן, התוסף שלך יעשה משהו מעניין יותר.

המחלקה [UnitarySynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin) מגדירה את הממשק והחוזה עבור תוספי סינתזת יוניטרי.
השיטה העיקרית היא [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin#run),
שמקבלת כקלט מערך Numpy המאחסן מטריצה יוניטרית
ומחזירה [DAGCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.dagcircuit.DAGCircuit) המייצג את ה-Circuit המסונתז מאותה מטריצה יוניטרית.
בנוסף לשיטת `run`, ישנן מספר שיטות property שיש להגדיר.
ראה [UnitarySynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin) לתיעוד כל המאפיינים הנדרשים.

בואו ניצור את תת-המחלקה UnitarySynthesisPlugin שלנו:

In [5]:
from qiskit.transpiler.passes.synthesis import unitary_synthesis_plugin_names

unitary_synthesis_plugin_names()

['aqc', 'clifford', 'default', 'gridsynth', 'sk']

אם אתה מוצא שהקלטים הזמינים לשיטת [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin#run)
אינם מספיקים למטרותיך, אנא [פתח issue](https://github.com/Qiskit/qiskit/issues/new/choose) שמסביר את הדרישות שלך. שינויים בממשק התוסף, כגון הוספת קלטים אופציונליים נוספים, יתבצעו בצורה תואמת לאחור כדי שלא יידרשו שינויים מתוספים קיימים.

> **Note:** כל השיטות עם הקידומת `supports_` שמורות במחלקה נגזרת של `UnitarySynthesisPlugin` כחלק מהממשק. אין להגדיר שיטות `supports_*` מותאמות אישית בתת-מחלקה שאינן מוגדרות במחלקה המופשטת.

כעת נחשוף את התוסף על ידי הוספת entry point במטא-נתונים של חבילת Python שלנו.
כאן אנו מניחים שהמחלקה שהגדרנו חשופה במודול בשם `my_qiskit_plugin`, לדוגמה על ידי ייבואה בקובץ `__init__.py` של מודול `my_qiskit_plugin`.
נערוך את קובץ `pyproject.toml`, ‏`setup.cfg`, או `setup.py` של החבילה שלנו:

<Tabs>
  <TabItem value="package-table-toml" label="pyproject.toml" default>
    ```toml
    [project.entry-points."qiskit.unitary_synthesis"]
    "my_unitary_synthesis" = "my_qiskit_plugin:MyUnitarySynthesisPlugin"
    ```
  </TabItem>
  <TabItem value="package-table-cfg" label="setup.cfg">
    ```ini
    [options.entry_points]
    qiskit.unitary_synthesis =
        my_unitary_synthesis = my_qiskit_plugin:MyUnitarySynthesisPlugin
    ```
  </TabItem>
  <TabItem value="package-table-py" label="setup.py">
    ```python
    from setuptools import setup

    setup(
        # ...,
        entry_points={
            'qiskit.unitary_synthesis': [
                'my_unitary_synthesis = my_qiskit_plugin:MyUnitarySynthesisPlugin',
            ]
        }
    )
    ```
  </TabItem>
</Tabs>

כמו לפני כן, אם הפרויקט שלך משתמש ב-`setup.cfg` או `setup.py` במקום `pyproject.toml`, ראה את [תיעוד setuptools](https://setuptools.pypa.io/en/latest/userguide/entry_point.html) כיצד להתאים שורות אלו למצב שלך.

כדי לבדוק שהתוסף שלך מזוהה בהצלחה על ידי Qiskit, התקן את חבילת התוסף שלך ועקוב אחר ההוראות ב-[תוספי Transpiler](transpiler-plugins#list-available-unitary-synthesis-plugins) לרשימת התוספים המותקנים, וודא שהתוסף שלך מופיע ברשימה:

In [6]:
from qiskit.synthesis import synth_clifford_bm
from qiskit.transpiler.passes.synthesis.plugin import HighLevelSynthesisPlugin


class MyCliffordSynthesisPlugin(HighLevelSynthesisPlugin):
    def run(
        self,
        high_level_object,
        coupling_map=None,
        target=None,
        qubits=None,
        **options,
    ) -> QuantumCircuit:
        if high_level_object.num_qubits <= 3:
            return synth_clifford_bm(high_level_object)
        else:
            return None

This plugin synthesizes objects of type [Clifford](/docs/api/qiskit/qiskit.quantum_info.Clifford) that have
at most 3 qubits, using the `synth_clifford_bm` method.

Now, we expose the plugin by adding an entry point in our Python package metadata.
Here, we assume that the class we defined is exposed in a module called `my_qiskit_plugin`, for example by being imported in the `__init__.py` file of the `my_qiskit_plugin` module.
We edit the `pyproject.toml`, `setup.cfg`, or `setup.py` file of our package:

<Tabs>
  <TabItem value="package-table-toml" label="pyproject.toml" default>
    ```toml
    [project.entry-points."qiskit.synthesis"]
    "clifford.my_clifford_synthesis" = "my_qiskit_plugin:MyCliffordSynthesisPlugin"
    ```
  </TabItem>
  <TabItem value="package-table-cfg" label="setup.cfg">
    ```ini
    [options.entry_points]
    qiskit.synthesis =
        clifford.my_clifford_synthesis = my_qiskit_plugin:MyCliffordSynthesisPlugin
    ```
  </TabItem>
  <TabItem value="package-table-py" label="setup.py">
    ```python
    from setuptools import setup

    setup(
        # ...,
        entry_points={
            'qiskit.synthesis': [
                'clifford.my_clifford_synthesis = my_qiskit_plugin:MyCliffordSynthesisPlugin',
            ]
        }
    )
    ```
  </TabItem>
</Tabs>

The `name` consists of two parts separated by a dot (`.`):
- The name of the type of [Operation](/docs/api/qiskit/qiskit.circuit.Operation) that the plugin synthesizes (in this case, `clifford`). Note that this string corresponds to the [`name`](/docs/api/qiskit/qiskit.circuit.Operation#name) attribute of the Operation class, and not the name of the class itself.
- The name of the plugin (in this case, `special`).

As before, if your project uses `setup.cfg` or `setup.py` instead of `pyproject.toml`, see the [setuptools documentation](https://setuptools.pypa.io/en/latest/userguide/entry_point.html) for how to adapt these lines for your situation.

To check that your plugin is successfully detected by Qiskit, install your plugin package and follow the instructions at [Transpiler plugins](transpiler-plugins#list-available-high-level-synthesis-plugins) for listing installed plugins, and ensure that your plugin appears in the list:

In [7]:
from qiskit.transpiler.passes.synthesis import (
    high_level_synthesis_plugin_names,
)

high_level_synthesis_plugin_names("clifford")

['ag', 'bm', 'default', 'greedy', 'layers', 'lnn', 'rb_default']

אם תוסף הדוגמה שלנו היה מותקן, אז השם `my_unitary_synthesis` היה מופיע ברשימה זו.

כדי להתאים לתוספי סינתזת יוניטרי שחושפים אפשרויות מרובות,
לממשק התוסף יש אפשרות למשתמשים לספק מילון תצורה חופשי.
הוא יועבר לשיטת `run` דרך ארגומנט מילת המפתח `options`. אם לתוסף שלך יש אפשרויות תצורה אלו, עליך לתעד אותן בבירור.

## דוגמה: יצירת תוסף סינתזה ברמה גבוהה

בדוגמה זו נצור תוסף סינתזה ברמה גבוהה שפשוט משתמש בפונקציה המובנית [synth_clifford_bm](https://docs.quantum.ibm.com/api/qiskit/synthesis#synth_clifford_bm) לסינתזת אופרטור Clifford.

המחלקה [HighLevelSynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin) מגדירה את הממשק והחוזה עבור תוספי סינתזה ברמה גבוהה. השיטה העיקרית היא [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin#run).
הארגומנט המיקומי `high_level_object` הוא [Operation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Operation) המייצג את האובייקט "ברמה גבוהה" שיש לסנתז. לדוגמה, הוא יכול להיות
[LinearFunction](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction) או
[Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford).
ארגומנטי מילת המפתח הבאים קיימים:
- `target` מציין את ה-Backend המטרה, ומאפשר לתוסף
לגשת לכל המידע הספציפי למטרה,
כגון מפת הצימוד, מערך ה-Gate הנתמך, וכן הלאה
- `coupling_map` מציין רק את מפת הצימוד, ומשמש רק כאשר `target` אינו מצוין.
- `qubits` מציין את רשימת ה-Qubit-ים שעליהם מוגדר האובייקט ברמה גבוהה, במקרה שהסינתזה מתבצעת על ה-Circuit הפיזי.
ערך של ``None`` מציין שהפריסה טרם נבחרה וה-Qubit-ים הפיזיים ב-Backend המטרה או במפת הצימוד שעליהם פועלת פעולה זו טרם נקבעו.
- `options`, מילון תצורה חופשי לאפשרויות ספציפיות לתוסף. אם לתוסף שלך יש אפשרויות תצורה אלו
עליך לתעד אותן בבירור.

שיטת `run` מחזירה [QuantumCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit)
המייצג את ה-Circuit המסונתז מאותו אובייקט ברמה גבוהה.
מותר גם להחזיר `None`, שמציין שהתוסף אינו מסוגל לסנתז את האובייקט ברמה הגבוהה שניתן.
הסינתזה בפועל של אובייקטים ברמה גבוהה מתבצעת על ידי
מעבר ה-Transpiler [HighLevelSynthesis](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.HighLevelSynthesis).

בנוסף לשיטת `run`, ישנן מספר שיטות property שיש להגדיר.
ראה [HighLevelSynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin) לתיעוד כל המאפיינים הנדרשים.

בואו נגדיר את תת-המחלקה HighLevelSynthesisPlugin שלנו: